# Functions and Mappings — Exercises

8 graded exercises covering every major section of [notes.md](notes.md) and [theory.ipynb](theory.ipynb).

**Format**: Each exercise has:
- 🎯 **Problem** — real-world ML context with mathematical grounding
- 📝 **Scaffold** — fill in `# YOUR CODE HERE`
- ✅ **Solution** — in a separate cell, with assertions and explanations

| Level | Description | Exercises |
|-------|-------------|----------|
| ⭐ | Essential — core concepts | 1–3 |
| ⭐⭐ | Applied — real ML engineering | 4–6 |
| ⭐⭐⭐ | Advanced — proofs & deep theory | 7–8 |

| Section | Exercises Covered |
|---------|-------------------|
| Types of Functions (Injectivity/Surjectivity/Bijectivity) | 1 |
| Function Composition | 2 |
| Preimage Computation | 3 |
| Activation Functions (derivatives, fixed points, Lipschitz) | 4 |
| Lipschitz Constants (proofs and composition) | 5 |
| Higher-Order Functions (map, filter, reduce) | 6 |
| Inverse Functions and Pseudo-Inverse | 7 |
| Universal Approximation | 8 |

**References**:
- Rudin, *Principles of Mathematical Analysis*, 3rd ed. (1976) — continuity, differentiability
- Axler, *Linear Algebra Done Right*, 4th ed. (2024) — linear maps, injectivity/surjectivity
- Goodfellow, Bengio & Courville, *Deep Learning* (2016) — activation functions, universal approximation
- Vaswani et al., *Attention Is All You Need* (2017) — transformer as function composition
- Hornik, Stinchcombe & White, *Multilayer feedforward networks are universal approximators*, Neural Networks (1989)
- Cybenko, *Approximation by superpositions of a sigmoidal function*, Mathematics of Control, Signals and Systems (1989)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Set, Tuple, List, Dict, Any, Optional

np.random.seed(42)
np.set_printoptions(precision=8, suppress=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f'NumPy {np.__version__} | PyTorch {torch.__version__} — Ready!')
except ImportError:
    HAS_TORCH = False
    print(f'NumPy {np.__version__} — Ready! (PyTorch exercises will use NumPy fallbacks)')

---

## Exercise 1: Injectivity, Surjectivity, and Bijectivity ⭐

**Background**: A function f: A → B is:
- **Injective** (one-to-one) if f(a₁) = f(a₂) ⟹ a₁ = a₂ (no two inputs share an output)
- **Surjective** (onto) if ∀b ∈ B, ∃a ∈ A: f(a) = b (every output is achieved)
- **Bijective** if both injective and surjective (perfect one-to-one correspondence)

**Why it matters for AI**: Embeddings should ideally be injective (different tokens → different vectors).
Invertible networks (normalizing flows) require bijective layers. Understanding which properties hold
determines whether information is preserved or lost.

**Task**: For each of the following four functions, determine whether it is injective, surjective, 
bijective, or neither. If bijective, find the inverse function.

(a) f: ℝ → ℝ,  f(x) = 3x − 7
(b) f: ℝ → ℝ,  f(x) = x³ − x  
(c) f: ℝ → (0,∞),  f(x) = eˣ
(d) f: [0,∞) → [0,∞),  f(x) = x²

**Requirements**:
- Implement `classify_continuous(f, x_samples, codomain_min, codomain_max)` that numerically tests a function
- For each function, provide the mathematical reasoning (in comments) AND numerical verification
- For bijective functions, implement and verify the inverse

In [ ]:
# Exercise 1: Your Solution

def classify_continuous(f: Callable, x_samples: np.ndarray,
                        codomain_min: float, codomain_max: float) -> dict:
    """
    Numerically test injectivity and surjectivity of a continuous function.
    
    Args:
        f: the function to test
        x_samples: array of domain sample points
        codomain_min: lower bound of declared codomain
        codomain_max: upper bound of declared codomain
    
    Returns:
        dict with keys 'injective', 'surjective', 'bijective'
    """
    # YOUR CODE HERE
    # Hint for injectivity: check if f produces duplicate outputs on the samples
    # Hint for surjectivity: check if the range covers the full codomain
    pass

# (a) f(x) = 3x - 7, domain R, codomain R
# YOUR CODE HERE — classify and explain

# (b) f(x) = x^3 - x, domain R, codomain R
# YOUR CODE HERE — classify and explain

# (c) f(x) = e^x, domain R, codomain (0, inf)
# YOUR CODE HERE — classify and explain

# (d) f(x) = x^2, domain [0, inf), codomain [0, inf)
# YOUR CODE HERE — classify and explain

# For bijective functions, implement inverse and verify f_inv(f(x)) == x
# YOUR CODE HERE

In [ ]:
# Exercise 1: Solution ✅

def classify_continuous(f: Callable, x_samples: np.ndarray,
                        codomain_min: float, codomain_max: float,
                        tol: float = 1e-10) -> dict:
    """
    Numerically test injectivity and surjectivity of a continuous function.
    
    Injectivity test: sort outputs and check if any two are equal (within tolerance).
    Surjectivity test: check if output range approximately covers full codomain.
    """
    y = f(x_samples)
    
    # Injectivity: check for duplicate outputs
    y_sorted = np.sort(y)
    diffs = np.diff(y_sorted)
    injective = bool(np.all(diffs > tol))
    
    # Surjectivity: check if range covers codomain
    y_min, y_max = y.min(), y.max()
    surjective = (y_min <= codomain_min + abs(codomain_min) * 0.01 + 0.01 and
                  y_max >= codomain_max - abs(codomain_max) * 0.01 - 0.01)
    
    bijective = injective and surjective
    
    return {
        'injective': injective,
        'surjective': surjective,
        'bijective': bijective,
        'range_min': float(y_min),
        'range_max': float(y_max),
    }

print('Exercise 1: Injectivity, Surjectivity, Bijectivity')
print('=' * 65)

# ─── (a) f(x) = 3x − 7 on ℝ → ℝ ───
# Mathematical reasoning:
# Injective: f(a₁) = f(a₂) → 3a₁ - 7 = 3a₂ - 7 → a₁ = a₂ ✓
# Surjective: for any b ∈ ℝ, solve 3x - 7 = b → x = (b+7)/3 ∈ ℝ ✓
# Bijective ✓. Inverse: f⁻¹(y) = (y + 7) / 3
x_a = np.linspace(-1000, 1000, 100000)
result_a = classify_continuous(lambda x: 3*x - 7, x_a, -3000, 3000)
print(f'\n(a) f(x) = 3x − 7, ℝ → ℝ')
print(f'    Injective: {result_a["injective"]}  |  Surjective: {result_a["surjective"]}  |  BIJECTIVE: {result_a["bijective"]}')
print(f'    Proof: f(a₁)=f(a₂) ⟹ 3a₁-7=3a₂-7 ⟹ a₁=a₂ (injective)')
print(f'    Proof: ∀b∈ℝ, x=(b+7)/3 gives f(x)=b (surjective)')

# Verify inverse
f_a = lambda x: 3*x - 7
f_a_inv = lambda y: (y + 7) / 3
test_vals = np.array([-100, -1, 0, 1, 42, 999])
roundtrip = f_a_inv(f_a(test_vals))
assert np.allclose(roundtrip, test_vals), 'Inverse failed!'
print(f'    Inverse: f⁻¹(y) = (y+7)/3  |  Round-trip verified ✓')

# ─── (b) f(x) = x³ − x on ℝ → ℝ ───
# Mathematical reasoning:
# NOT injective: f(0) = 0, f(1) = 0, f(-1) = 0 — three inputs map to 0
# Surjective: as x→±∞, x³−x→±∞, so by IVT all reals are achieved ✓
# Therefore: surjective but NOT injective
x_b = np.linspace(-100, 100, 200000)
result_b = classify_continuous(lambda x: x**3 - x, x_b, -100**3, 100**3)
print(f'\n(b) f(x) = x³ − x, ℝ → ℝ')
print(f'    Injective: {result_b["injective"]}  |  Surjective: {result_b["surjective"]}  |  Bijective: {result_b["bijective"]}')
print(f'    Counterexample: f(0)={0**3-0}, f(1)={1**3-1}, f(-1)={(-1)**3-(-1)} — all map to 0')
print(f'    Surjective: lim x→±∞ (x³−x) = ±∞, so by IVT, all reals achieved')

# ─── (c) f(x) = eˣ on ℝ → (0,∞) ───
# Mathematical reasoning:
# Injective: f(a₁) = f(a₂) → eᵃ¹ = eᵃ² → a₁ = a₂ (exp is strictly increasing) ✓
# Surjective onto (0,∞): for any b > 0, x = ln(b) gives eˣ = b ✓
# Bijective ✓. Inverse: f⁻¹(y) = ln(y)
x_c = np.linspace(-20, 20, 100000)
result_c = classify_continuous(np.exp, x_c, 0.0001, np.exp(20))
print(f'\n(c) f(x) = eˣ, ℝ → (0,∞)')
print(f'    Injective: {result_c["injective"]}  |  Surjective: {result_c["surjective"]}  |  BIJECTIVE: {result_c["bijective"]}')
print(f'    Proof: eˣ strictly increasing ⟹ injective')
print(f'    Proof: ∀b>0, x=ln(b) gives eˣ=b (surjective onto (0,∞))')
print(f'    NOTE: NOT surjective onto ℝ (negative values never achieved)')

# Verify inverse
f_c_inv = np.log
test_pos = np.array([0.01, 0.5, 1, 2.71828, 100])
assert np.allclose(f_c_inv(np.exp(test_pos)), test_pos), 'Inverse failed!'
print(f'    Inverse: f⁻¹(y) = ln(y)  |  Round-trip verified ✓')

# ─── (d) f(x) = x² on [0,∞) → [0,∞) ───
# Mathematical reasoning:
# Injective on [0,∞): f(a₁)=f(a₂) → a₁²=a₂² → |a₁|=|a₂| → a₁=a₂ (since both ≥ 0) ✓
# NOTE: on full ℝ, NOT injective: f(2)=f(-2)=4
# Surjective onto [0,∞): for any b≥0, x=√b gives f(x)=b ✓
# Bijective ✓ on [0,∞)→[0,∞). Inverse: f⁻¹(y) = √y
x_d = np.linspace(0, 1000, 100000)
result_d = classify_continuous(lambda x: x**2, x_d, 0, 1000**2)
print(f'\n(d) f(x) = x², [0,∞) → [0,∞)')
print(f'    Injective: {result_d["injective"]}  |  Surjective: {result_d["surjective"]}  |  BIJECTIVE: {result_d["bijective"]}')
print(f'    Key: restricting domain to [0,∞) makes x² injective!')
print(f'    On full ℝ it would NOT be injective: f(2)=f(-2)=4')

f_d_inv = np.sqrt
test_nonneg = np.array([0, 1, 4, 9, 100, 10000])
assert np.allclose(f_d_inv(test_nonneg**2), test_nonneg), 'Inverse failed!'
print(f'    Inverse: f⁻¹(y) = √y  |  Round-trip verified ✓')

print(f'\n{"="*65}')
print('Summary:')
print('  (a) 3x−7:  Bijective ← linear with nonzero slope')
print('  (b) x³−x:  Surjective only ← has critical points where derivative=0')
print('  (c) eˣ:    Bijective ℝ→(0,∞) ← strictly increasing, range=(0,∞)')
print('  (d) x²:    Bijective [0,∞)→[0,∞) ← domain restriction removes the f(x)=f(-x) problem')
print('\nAll tests passed ✓')

---

## Exercise 2: Function Composition ⭐

**Background**: Given f: A → B and g: B → C, the composition (g ∘ f): A → C is defined by (g ∘ f)(a) = g(f(a)).
Composition is the fundamental operation in neural networks — a transformer with L layers is f_L ∘ f_{L-1} ∘ ... ∘ f_1 ∘ E.

Composition is **associative** (h ∘ (g ∘ f) = (h ∘ g) ∘ f) but generally **NOT commutative** (g ∘ f ≠ f ∘ g).

**Task**: 
- Define f(x) = 2x + 1 and g(x) = x²
- Compute f ∘ g, g ∘ f, f ∘ f, g ∘ g both symbolically (in comments) and numerically
- Verify (g ∘ f)(3) = g(f(3)) step by step
- Demonstrate that f ∘ g ≠ g ∘ f with a concrete counterexample
- Verify associativity with a third function h(x) = sin(x)

**Requirements**:
- Implement a `compose(f, g)` function that returns a new function representing f ∘ g
- Plot all four compositions on [-3, 3]
- Show the non-commutativity clearly

In [ ]:
# Exercise 2: Your Solution

def compose(f: Callable, g: Callable) -> Callable:
    """Return the composition f ∘ g, i.e. x ↦ f(g(x))."""
    # YOUR CODE HERE
    pass

# Define base functions
# f(x) = 2x + 1
# g(x) = x^2

# Compute these four compositions and evaluate at x = 3:
# (f ∘ g)(x) = f(g(x)) = f(x²) = ?
# (g ∘ f)(x) = g(f(x)) = g(2x+1) = ?
# (f ∘ f)(x) = f(f(x)) = f(2x+1) = ?
# (g ∘ g)(x) = g(g(x)) = g(x²) = ?

# YOUR CODE HERE

# Verify step by step: (g ∘ f)(3) = g(f(3))
# YOUR CODE HERE

# Show f ∘ g ≠ g ∘ f
# YOUR CODE HERE

# Verify associativity with h(x) = sin(x)
# h ∘ (g ∘ f) == (h ∘ g) ∘ f ?
# YOUR CODE HERE

In [ ]:
# Exercise 2: Solution ✅

def compose(f: Callable, g: Callable) -> Callable:
    """Return the composition f ∘ g, i.e. x ↦ f(g(x))."""
    return lambda x: f(g(x))

# Define base functions
f = lambda x: 2*x + 1       # f(x) = 2x + 1
g = lambda x: x**2           # g(x) = x²

# ── Four compositions ──
f_of_g = compose(f, g)   # (f ∘ g)(x) = f(x²) = 2x² + 1
g_of_f = compose(g, f)   # (g ∘ f)(x) = g(2x+1) = (2x+1)²
f_of_f = compose(f, f)   # (f ∘ f)(x) = f(2x+1) = 2(2x+1) + 1 = 4x + 3
g_of_g = compose(g, g)   # (g ∘ g)(x) = g(x²) = x⁴

print('Exercise 2: Function Composition')
print('=' * 60)

# Symbolic derivations (verified numerically)
x_test = np.array([-2, -1, 0, 1, 2, 3])
print('Symbolic forms:')
print('  (f ∘ g)(x) = f(g(x)) = f(x²) = 2x² + 1')
print('  (g ∘ f)(x) = g(f(x)) = g(2x+1) = (2x+1)² = 4x² + 4x + 1')
print('  (f ∘ f)(x) = f(f(x)) = f(2x+1) = 2(2x+1) + 1 = 4x + 3')
print('  (g ∘ g)(x) = g(g(x)) = g(x²) = (x²)² = x⁴')

# Verify numerically
print(f'\nNumerical verification at x = {x_test}:')
print(f'  (f ∘ g)(x) = {f_of_g(x_test)}')
print(f'  2x² + 1    = {2*x_test**2 + 1}')
assert np.allclose(f_of_g(x_test), 2*x_test**2 + 1), 'f∘g failed'
print(f'  Match ✓')

print(f'\n  (g ∘ f)(x) = {g_of_f(x_test)}')
print(f'  (2x+1)²    = {(2*x_test+1)**2}')
assert np.allclose(g_of_f(x_test), (2*x_test+1)**2), 'g∘f failed'
print(f'  Match ✓')

# ── Step-by-step verification: (g ∘ f)(3) ──
print(f'\nStep-by-step: (g ∘ f)(3)')
step1 = f(3)          # f(3) = 2(3) + 1 = 7
step2 = g(step1)      # g(7) = 7² = 49
direct = g_of_f(3)    # (g ∘ f)(3) should be 49
print(f'  Step 1: f(3) = 2(3) + 1 = {step1}')
print(f'  Step 2: g(f(3)) = g({step1}) = {step1}² = {step2}')
print(f'  Direct: (g ∘ f)(3) = {direct}')
assert step2 == direct, 'Step-by-step != direct'
print(f'  g(f(3)) == (g ∘ f)(3) ✓')

# ── Non-commutativity: f ∘ g ≠ g ∘ f ──
print(f'\nNon-commutativity demonstration:')
print(f'  (f ∘ g)(3) = f(g(3)) = f(9) = 2(9)+1 = {f_of_g(3)}')
print(f'  (g ∘ f)(3) = g(f(3)) = g(7) = 7²    = {g_of_f(3)}')
print(f'  19 ≠ 49 → f ∘ g ≠ g ∘ f  (composition is NOT commutative) ✓')
assert f_of_g(3) != g_of_f(3), 'They should differ!'

# ── Associativity: h ∘ (g ∘ f) == (h ∘ g) ∘ f ──
h = np.sin
h_of_gf = compose(h, g_of_f)           # h ∘ (g ∘ f)
hg_of_f = compose(compose(h, g), f)    # (h ∘ g) ∘ f

x_assoc = np.linspace(-5, 5, 10000)
assert np.allclose(h_of_gf(x_assoc), hg_of_f(x_assoc)), 'Associativity failed!'
print(f'\nAssociativity: h ∘ (g ∘ f) == (h ∘ g) ∘ f')
print(f'  h = sin, tested on 10000 points in [-5, 5]')
print(f'  Max difference: {np.max(np.abs(h_of_gf(x_assoc) - hg_of_f(x_assoc))):.2e}')
print(f'  Associativity verified ✓')

# ── Visualization ──
x_plot = np.linspace(-3, 3, 500)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

compositions = [
    ('f ∘ g = 2x² + 1', f_of_g),
    ('g ∘ f = (2x+1)²', g_of_f),
    ('f ∘ f = 4x + 3', f_of_f),
    ('g ∘ g = x⁴', g_of_g),
]

for ax, (title, fn) in zip(axes, compositions):
    ax.plot(x_plot, fn(x_plot), 'b-', linewidth=2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_xlabel('x')

plt.suptitle('Four Compositions of f(x)=2x+1 and g(x)=x²', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey insight for AI: layer order matters! swapping layers ≠ same model')
print('All tests passed ✓')

---

## Exercise 3: Preimage Computation ⭐

**Background**: The preimage f⁻¹(T) = {a ∈ A | f(a) ∈ T} is the set of all inputs mapping into T.
This does NOT require f to be invertible — preimage of a *set* always exists.

Preimage preserves all set operations exactly:
- f⁻¹(T₁ ∪ T₂) = f⁻¹(T₁) ∪ f⁻¹(T₂)
- f⁻¹(T₁ ∩ T₂) = f⁻¹(T₁) ∩ f⁻¹(T₂)  
- f⁻¹(B \ T) = A \ f⁻¹(T)

Image does NOT preserve intersection: f(S₁ ∩ S₂) ⊆ f(S₁) ∩ f(S₂) but equality can fail.

**Task**: Define f: ℝ → ℝ by f(x) = x². Compute:
1. f⁻¹({4}) — all x where x² = 4
2. f⁻¹({−1}) — all x where x² = −1
3. f⁻¹([0, 4]) — all x where 0 ≤ x² ≤ 4
4. f⁻¹([1, 9]) — all x where 1 ≤ x² ≤ 9
5. Verify De Morgan: f⁻¹(ℝ \ [1, 4]) = ℝ \ f⁻¹([1, 4])
6. Find a concrete example where f(S₁ ∩ S₂) ⊊ f(S₁) ∩ f(S₂) (strict subset)

**Requirements**:
- Implement `preimage_numerical(f, x_domain, target_set_check)` for continuous functions
- Visualize the preimage sets on a number line
- Verify all set operation properties

In [ ]:
# Exercise 3: Your Solution

def preimage_numerical(f: Callable, x_domain: np.ndarray,
                       target_check: Callable) -> np.ndarray:
    """
    Numerically compute preimage: {x in x_domain | target_check(f(x)) is True}.
    
    Args:
        f: the function
        x_domain: array of domain sample points
        target_check: function returning True if f(x) is in the target set
    
    Returns:
        array of x values in the preimage
    """
    # YOUR CODE HERE
    pass

# f(x) = x^2

# 1. f^{-1}({4})
# YOUR CODE HERE

# 2. f^{-1}({-1})
# YOUR CODE HERE

# 3. f^{-1}([0, 4])
# YOUR CODE HERE

# 4. f^{-1}([1, 9])
# YOUR CODE HERE

# 5. De Morgan verification
# YOUR CODE HERE

# 6. Show f(S1 ∩ S2) ⊊ f(S1) ∩ f(S2) for some S1, S2
# YOUR CODE HERE

In [ ]:
# Exercise 3: Solution ✅

def preimage_numerical(f: Callable, x_domain: np.ndarray,
                       target_check: Callable) -> np.ndarray:
    """Numerically compute preimage: {x in x_domain | target_check(f(x))}."""
    y = f(x_domain)
    mask = target_check(y)
    return x_domain[mask]

f = lambda x: x**2
x_domain = np.linspace(-5, 5, 100000)

print('Exercise 3: Preimage of f(x) = x²')
print('=' * 60)

# ─── 1. f⁻¹({4}) — all x where x² = 4 ───
# Analytical: x² = 4 → x = ±2
tol = 0.01
pre_4 = preimage_numerical(f, x_domain, lambda y: np.abs(y - 4) < tol)
print(f'\n1. f⁻¹({{4}}) ≈ {{{pre_4.min():.3f}, {pre_4.max():.3f}}}')
print(f'   Analytical: {{-2, 2}}')
print(f'   Points found near ±2: {len(pre_4)} sample points')

# ─── 2. f⁻¹({−1}) — all x where x² = −1 ───
# Analytical: no real solution, so f⁻¹({-1}) = ∅
pre_neg1 = preimage_numerical(f, x_domain, lambda y: np.abs(y - (-1)) < tol)
print(f'\n2. f⁻¹({{-1}}) = ∅ (empty set)')
print(f'   Points found: {len(pre_neg1)} ← no real number squares to -1')

# ─── 3. f⁻¹([0, 4]) — all x where 0 ≤ x² ≤ 4 ───
# Analytical: |x| ≤ 2, so x ∈ [-2, 2]
pre_04 = preimage_numerical(f, x_domain, lambda y: (y >= 0) & (y <= 4))
print(f'\n3. f⁻¹([0, 4]) ≈ [{pre_04.min():.3f}, {pre_04.max():.3f}]')
print(f'   Analytical: [-2, 2]')

# ─── 4. f⁻¹([1, 9]) — all x where 1 ≤ x² ≤ 9 ───
# Analytical: 1 ≤ x² ≤ 9 means |x| ∈ [1, 3], so x ∈ [-3, -1] ∪ [1, 3]
pre_19 = preimage_numerical(f, x_domain, lambda y: (y >= 1) & (y <= 9))
neg_part = pre_19[pre_19 < 0]
pos_part = pre_19[pre_19 > 0]
print(f'\n4. f⁻¹([1, 9]) ≈ [{neg_part.min():.3f}, {neg_part.max():.3f}] ∪ [{pos_part.min():.3f}, {pos_part.max():.3f}]')
print(f'   Analytical: [-3, -1] ∪ [1, 3]')

# ─── 5. De Morgan: f⁻¹(ℝ \ [1, 4]) = ℝ \ f⁻¹([1, 4]) ───
pre_14 = preimage_numerical(f, x_domain, lambda y: (y >= 1) & (y <= 4))
pre_complement = preimage_numerical(f, x_domain, lambda y: ~((y >= 1) & (y <= 4)))

# ℝ \ f⁻¹([1,4]) should be the same as f⁻¹(ℝ \ [1,4])
domain_minus_pre14 = np.setdiff1d(
    np.round(x_domain, 6),
    np.round(pre_14, 6)
)

print(f'\n5. De Morgan verification:')
print(f'   f⁻¹([1,4]) has {len(pre_14)} sample points')
print(f'   f⁻¹(ℝ \\ [1,4]) has {len(pre_complement)} sample points')
print(f'   domain \\ f⁻¹([1,4]) has {len(domain_minus_pre14)} sample points')
print(f'   |f⁻¹(complement)| ≈ |domain \\ f⁻¹(set)|: {abs(len(pre_complement) - len(domain_minus_pre14)) <= 1} ✓')

# ─── 6. f(S₁ ∩ S₂) ⊊ f(S₁) ∩ f(S₂) — strict inclusion ───
# Use f(x) = x² with S₁ = {-2, 1} and S₂ = {2, 3}
# S₁ ∩ S₂ = ∅, so f(S₁ ∩ S₂) = ∅
# f(S₁) = {4, 1}, f(S₂) = {4, 9}
# f(S₁) ∩ f(S₂) = {4}
# ∅ ⊊ {4} — strict subset!
S1 = {-2, 1}
S2 = {2, 3}
S1_inter_S2 = S1 & S2
f_of_inter = {x**2 for x in S1_inter_S2}
f_S1 = {x**2 for x in S1}
f_S2 = {x**2 for x in S2}
f_S1_inter_f_S2 = f_S1 & f_S2

print(f'\n6. Image does NOT preserve intersection:')
print(f'   S₁ = {S1}, S₂ = {S2}')
print(f'   S₁ ∩ S₂ = {S1_inter_S2}')
print(f'   f(S₁ ∩ S₂) = {f_of_inter}')
print(f'   f(S₁) = {f_S1}, f(S₂) = {f_S2}')
print(f'   f(S₁) ∩ f(S₂) = {f_S1_inter_f_S2}')
print(f'   f(S₁ ∩ S₂) ⊊ f(S₁) ∩ f(S₂): {f_of_inter.issubset(f_S1_inter_f_S2) and f_of_inter != f_S1_inter_f_S2} ✓')
print(f'   ∅ ⊊ {{4}} — the value 4 appears in both f(S₁) and f(S₂)')
print(f'   because f(-2)=4 and f(2)=4, but -2 ∈ S₁ and 2 ∈ S₂ (different inputs!)')

# ── Visualization ──
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

queries = [
    ('f⁻¹({4})', lambda y: np.abs(y - 4) < 0.1, '{4}'),
    ('f⁻¹({−1}) = ∅', lambda y: np.abs(y - (-1)) < 0.1, '{-1}'),
    ('f⁻¹([0, 4]) = [-2, 2]', lambda y: (y >= 0) & (y <= 4), '[0, 4]'),
    ('f⁻¹([1, 9]) = [-3,-1]∪[1,3]', lambda y: (y >= 1) & (y <= 9), '[1, 9]'),
]

for ax, (title, check, target_label) in zip(axes.flat, queries):
    y_domain = f(x_domain)
    mask = check(y_domain)
    
    ax.plot(x_domain, y_domain, 'b-', alpha=0.3, linewidth=1)
    ax.scatter(x_domain[mask], y_domain[mask], c='red', s=1, alpha=0.5, label='preimage')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('x (domain)')
    ax.set_ylabel('f(x) = x²')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('Preimage Visualization for f(x) = x²', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nAll tests passed ✓')

---

## Exercise 4: Activation Functions ⭐⭐

**Background**: Activation functions are the nonlinear functions applied elementwise to break linearity 
in neural networks. Without them, a composition of linear layers is just a single linear layer — depth 
gives no expressive benefit.

Key properties of activation functions:
- **Derivative**: determines gradient flow; vanishing derivatives → vanishing gradients
- **Fixed points**: solutions to f(x) = x; dynamically stable states
- **Injectivity**: determines whether information is preserved
- **Lipschitz constant**: bounds how fast the function changes; affects training stability

**Reference**: Goodfellow et al., *Deep Learning* (2016), Ch. 6.3; Nair & Hinton (2010) for ReLU;
Hendrycks & Gimpel (2016) for GELU; Shazeer (2020) for SwiGLU/GLU variants.

**Task**: For each of ReLU, sigmoid σ(x) = 1/(1+e⁻ˣ), and tanh(x):
1. Implement the function and its analytical derivative
2. Find all fixed points f(x) = x numerically
3. Determine if injective/surjective as ℝ → ℝ
4. Compute the Lipschitz constant (= max|f'(x)|)
5. Plot f(x) and f'(x) side by side

**Requirements**:
- Implement derivatives from the analytical formulas (not numerical differentiation)
- Find fixed points using Newton's method or bisection on g(x) = f(x) − x
- Verify Lipschitz constant matches theoretical value

In [ ]:
# Exercise 4: Your Solution

# (a) ReLU: f(x) = max(0, x)
def relu(x):
    # YOUR CODE HERE
    pass

def relu_derivative(x):
    # YOUR CODE HERE — use: 1 if x > 0, 0 if x < 0
    pass

# (b) Sigmoid: σ(x) = 1/(1+e^{-x})
def sigmoid(x):
    # YOUR CODE HERE
    pass

def sigmoid_derivative(x):
    # YOUR CODE HERE — use: σ(x)(1 − σ(x))
    pass

# (c) Tanh: tanh(x) = (e^x − e^{-x})/(e^x + e^{-x})
def tanh_fn(x):
    # YOUR CODE HERE
    pass

def tanh_derivative(x):
    # YOUR CODE HERE — use: 1 − tanh²(x)
    pass

# For each: find fixed points, determine injectivity, compute Lipschitz constant
# YOUR CODE HERE

In [ ]:
# Exercise 4: Solution ✅

# ══════════════════════════════════════════════════════════════════
# Implementations
# ══════════════════════════════════════════════════════════════════

# (a) ReLU
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return np.where(x > 0, 1.0, 0.0)  # undefined at 0, use 0 by convention

# (b) Sigmoid: σ(x) = 1/(1+e^{-x})
# Derivative: σ'(x) = σ(x)(1 − σ(x))
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1.0 - s)

# (c) Tanh
# Derivative: tanh'(x) = 1 − tanh²(x)  , max at x=0 where tanh'(0)=1
def tanh_fn(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1.0 - np.tanh(x)**2

# ══════════════════════════════════════════════════════════════════
# Analysis
# ══════════════════════════════════════════════════════════════════

x = np.linspace(-6, 6, 100000)

activations = [
    ('ReLU', relu, relu_derivative),
    ('Sigmoid', sigmoid, sigmoid_derivative),
    ('Tanh', tanh_fn, tanh_derivative),
]

print('Exercise 4: Activation Function Analysis')
print('=' * 70)

for name, fn, dfn in activations:
    y = fn(x)
    dy = dfn(x)
    
    # ── Fixed points: f(x) = x ──
    # Find where |f(x) - x| is small
    g = np.abs(y - x)
    fixed_mask = g < 0.005
    if np.any(fixed_mask):
        # Cluster nearby fixed points
        fixed_candidates = x[fixed_mask]
        # Take representative points (cluster centers)
        fixed_points = []
        for fp in fixed_candidates:
            if not fixed_points or abs(fp - fixed_points[-1]) > 0.05:
                fixed_points.append(fp)
    else:
        fixed_points = []
    
    # ── Injectivity ──
    y_sorted = np.sort(y)
    diffs = np.diff(y_sorted)
    injective = bool(np.all(diffs > 1e-12))
    
    # ── Surjectivity (onto ℝ) ──
    surjective = (y.min() < -5.9 and y.max() > 5.9)  # approaches ±∞
    
    # ── Lipschitz constant = max|f'(x)| ──
    lip_const = np.max(np.abs(dy))
    
    print(f'\n{name}:')
    print(f'  Range: [{y.min():.4f}, {y.max():.4f}]')
    print(f'  Fixed points (f(x)=x): {[round(fp, 4) for fp in fixed_points]}')
    print(f'  Injective (ℝ → ℝ): {injective}')
    print(f'  Surjective (onto ℝ): {surjective}')
    print(f'  Lipschitz constant (max|f\'(x)|): {lip_const:.4f}')

# ── Detailed analysis per activation ──
print(f'\n{"="*70}')
print('Detailed Analysis:')

print(f'\nReLU:')
print(f'  f(x) = max(0,x)')
print(f'  f\'(x) = 1 if x>0, 0 if x<0, undefined at x=0')
print(f'  Fixed points: x=0 (only point where max(0,x) = x on negative side), and all x>0')
print(f'  NOT injective: f(-1) = f(-2) = 0 (all negatives map to 0)')
print(f'  NOT surjective onto ℝ: range = [0,∞), negative outputs unreachable')
print(f'  Lipschitz constant: L = 1 (|f(x)-f(y)| ≤ |x-y| always)')
print(f'  Dead neuron problem: if input always < 0, gradient = 0, neuron never updates')

print(f'\nSigmoid σ(x):')
print(f'  σ(x) = 1/(1+e^(-x))')
print(f'  σ\'(x) = σ(x)(1-σ(x)), maximum at x=0 where σ\'(0) = 0.25')
print(f'  Fixed points: one fixed point near x ≈ 0.5 (where σ(x)=x, solve numerically)')
print(f'  Injective: YES (strictly increasing)')
print(f'  NOT surjective onto ℝ: range = (0,1)')
print(f'  Lipschitz constant: L = 0.25 (max derivative)')
print(f'  Vanishing gradient: for |x| >> 0, σ\'(x) ≈ 0 → gradient vanishes')

# Verify sigmoid fixed point
from scipy.optimize import brentq
fp_sigmoid = brentq(lambda x: sigmoid(np.array([x]))[0] - x, 0, 1)
print(f'  Exact fixed point: x ≈ {fp_sigmoid:.6f} (σ({fp_sigmoid:.4f}) = {sigmoid(np.array([fp_sigmoid]))[0]:.6f})')

print(f'\nTanh:')
print(f'  tanh(x) = (eˣ - e⁻ˣ)/(eˣ + e⁻ˣ)')
print(f'  tanh\'(x) = 1 - tanh²(x), maximum at x=0 where tanh\'(0) = 1')
print(f'  Fixed points: x = 0 (tanh(0) = 0 = 0)')
print(f'  Injective: YES (strictly increasing)')
print(f'  NOT surjective onto ℝ: range = (-1, 1)')
print(f'  Lipschitz constant: L = 1 (max derivative at x=0)')
print(f'  Better than sigmoid: zero-centered output, larger gradient at origin')

# ══════════════════════════════════════════════════════════════════
# Visualization
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
x_plot = np.linspace(-6, 6, 1000)

for i, (name, fn, dfn) in enumerate(activations):
    y_plot = fn(x_plot)
    dy_plot = dfn(x_plot)
    
    # Plot function
    axes[i, 0].plot(x_plot, y_plot, 'b-', linewidth=2, label=f'{name}(x)')
    axes[i, 0].plot(x_plot, x_plot, 'k--', alpha=0.3, linewidth=1, label='y = x')
    # Mark fixed points
    g_vals = np.abs(fn(x_plot) - x_plot)
    fp_mask = g_vals < 0.03
    if np.any(fp_mask):
        axes[i, 0].scatter(x_plot[fp_mask][::10], y_plot[fp_mask][::10], 
                          c='red', s=30, zorder=5, label='fixed points')
    axes[i, 0].set_title(f'{name}(x)', fontweight='bold')
    axes[i, 0].legend(fontsize=8)
    axes[i, 0].grid(True, alpha=0.3)
    axes[i, 0].axhline(0, color='k', linewidth=0.5)
    axes[i, 0].axvline(0, color='k', linewidth=0.5)
    
    # Plot derivative
    axes[i, 1].plot(x_plot, dy_plot, 'r-', linewidth=2, label=f"{name}'(x)")
    axes[i, 1].axhline(np.max(np.abs(dy_plot)), color='green', linestyle='--', 
                       alpha=0.5, label=f'Lipschitz L={np.max(np.abs(dy_plot)):.2f}')
    axes[i, 1].set_title(f"{name}'(x) — derivative", fontweight='bold')
    axes[i, 1].legend(fontsize=8)
    axes[i, 1].grid(True, alpha=0.3)
    axes[i, 1].axhline(0, color='k', linewidth=0.5)
    axes[i, 1].axvline(0, color='k', linewidth=0.5)

plt.suptitle('Activation Functions: f(x) and f\'(x)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nAll tests passed ✓')

---

## Exercise 5: Lipschitz Constants ⭐⭐

**Background**: A function f: ℝⁿ → ℝᵐ is **L-Lipschitz** if ‖f(x) − f(y)‖ ≤ L‖x − y‖ for all x, y.
The Lipschitz constant bounds how fast f can change — it's essential for:
- **Adversarial robustness**: small input perturbation → bounded output change (Szegedy et al., 2014)
- **Wasserstein GAN**: discriminator must be 1-Lipschitz (Arjovsky et al., 2017)
- **Spectral normalization**: constrains weight matrix spectral norm (Miyato et al., 2018)
- **Training stability**: Lipschitz-bounded networks have controllable gradient magnitudes

**Key results to prove**:
(a) Every linear map f(x) = Ax has Lipschitz constant = ‖A‖₂ (spectral norm = largest singular value)
(b) Composition of L₁-Lipschitz and L₂-Lipschitz functions is (L₁·L₂)-Lipschitz
(c) A differentiable function with |f'(x)| ≤ L for all x is L-Lipschitz (via mean value theorem)

**Task**: 
- Implement a numerical Lipschitz constant estimator
- Verify all three properties with concrete examples
- Compute the Lipschitz constant of a 2-layer neural network

**Requirements**:
- For (a): construct random matrices, compute spectral norm, verify Lipschitz bound
- For (b): compose two functions, show L_composed ≤ L₁ · L₂
- For (c): verify using sigmoid (max derivative = 0.25, so 0.25-Lipschitz)

In [ ]:
# Exercise 5: Your Solution

def estimate_lipschitz_1d(f: Callable, x_samples: np.ndarray) -> float:
    """
    Estimate Lipschitz constant of f: R -> R by computing max |f(x)-f(y)|/|x-y|
    over all pairs of consecutive sample points.
    """
    # YOUR CODE HERE
    pass

def estimate_lipschitz_matrix(A: np.ndarray, n_samples: int = 10000) -> float:
    """
    Estimate Lipschitz constant of f(x) = Ax by sampling random unit vectors
    and computing max ||Ax||.
    """
    # YOUR CODE HERE
    pass

# (a) Show linear map Ax has Lipschitz constant = spectral norm ||A||_2
# YOUR CODE HERE

# (b) Show composition Lipschitz constant ≤ product of individual constants
# YOUR CODE HERE

# (c) Show bounded derivative implies Lipschitz
# YOUR CODE HERE

In [ ]:
# Exercise 5: Solution ✅

print('Exercise 5: Lipschitz Constants')
print('=' * 70)

# ══════════════════════════════════════════════════════════════════
# Helper: Estimate Lipschitz constant for 1D and matrix functions
# ══════════════════════════════════════════════════════════════════

def estimate_lipschitz_1d(f: Callable, x_samples: np.ndarray) -> float:
    """Estimate Lipschitz constant by max ratio |f(x)-f(y)|/|x-y| on consecutive points."""
    y = f(x_samples)
    dx = np.diff(x_samples)
    dy = np.diff(y)
    # Avoid division by zero
    nonzero = np.abs(dx) > 1e-15
    ratios = np.abs(dy[nonzero]) / np.abs(dx[nonzero])
    return float(np.max(ratios))

def estimate_lipschitz_matrix(A: np.ndarray, n_samples: int = 50000) -> float:
    """Estimate Lipschitz constant of f(x) = Ax by max ||Ax|| over unit vectors."""
    d = A.shape[1]
    # Sample random unit vectors
    X = np.random.randn(n_samples, d)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    # Compute ||Ax|| for each
    AX = (A @ X.T).T  # (n_samples, m)
    norms = np.linalg.norm(AX, axis=1)
    return float(np.max(norms))

# ══════════════════════════════════════════════════════════════════
# (a) Linear map f(x) = Ax has Lipschitz constant = spectral norm ||A||_2
# ══════════════════════════════════════════════════════════════════

print('\n(a) Linear map: Lipschitz constant = spectral norm')
print('-' * 60)

# Proof sketch (in comments):
# ||f(x) - f(y)|| = ||A(x-y)|| ≤ ||A||_2 · ||x-y||
# where ||A||_2 = max singular value = σ_max(A)
# The bound is tight: achieved when x-y = right singular vector for σ_max

np.random.seed(42)
for trial in range(3):
    m, n = np.random.choice([3, 4, 5], 2)
    A = np.random.randn(m, n)
    
    # Exact spectral norm via SVD
    singular_values = np.linalg.svd(A, compute_uv=False)
    spectral_norm = singular_values[0]
    
    # Estimated Lipschitz constant
    estimated_L = estimate_lipschitz_matrix(A, n_samples=100000)
    
    print(f'  A: {m}×{n} random matrix')
    print(f'    ||A||_2 (spectral norm) = {spectral_norm:.6f}')
    print(f'    Estimated Lipschitz     = {estimated_L:.6f}')
    print(f'    Ratio (should be ≤ 1):    {estimated_L / spectral_norm:.6f}')
    assert estimated_L <= spectral_norm * 1.01, 'Lipschitz > spectral norm!'
    print(f'    ✓ Estimated ≤ spectral norm (tight bound)')

# ══════════════════════════════════════════════════════════════════
# (b) Composition: L(g∘f) ≤ L(g) · L(f)
# ══════════════════════════════════════════════════════════════════

print(f'\n(b) Composition: L(g∘f) ≤ L(g) · L(f)')
print('-' * 60)

# Proof sketch:
# ||g(f(x)) - g(f(y))|| ≤ L_g · ||f(x) - f(y)|| ≤ L_g · L_f · ||x - y||

# Example: f(x) = sigmoid (L_f = 0.25), g(x) = 2x+1 (L_g = 2)
# g∘f should be at most 0.25 * 2 = 0.5 Lipschitz
f_lip = lambda x: 1.0 / (1.0 + np.exp(-x))  # sigmoid, L=0.25
g_lip = lambda x: 2*x + 1                     # linear, L=2
gf = lambda x: g_lip(f_lip(x))               # composition

x_dense = np.linspace(-10, 10, 500000)
L_f = estimate_lipschitz_1d(f_lip, x_dense)
L_g = estimate_lipschitz_1d(g_lip, x_dense)
L_gf = estimate_lipschitz_1d(gf, x_dense)

print(f'  f = sigmoid:   L_f  ≈ {L_f:.4f}  (theoretical: 0.25)')
print(f'  g = 2x + 1:    L_g  ≈ {L_g:.4f}  (theoretical: 2.0)')
print(f'  g ∘ f:          L_gf ≈ {L_gf:.4f}')
print(f'  L_f · L_g     = {L_f * L_g:.4f}')
print(f'  L_gf ≤ L_f·L_g: {L_gf <= L_f * L_g + 0.01} ✓')

# Another example: two random matrix multiplications
A = np.random.randn(3, 4)
B = np.random.randn(5, 3)
L_A = np.linalg.svd(A, compute_uv=False)[0]
L_B = np.linalg.svd(B, compute_uv=False)[0]
L_BA = np.linalg.svd(B @ A, compute_uv=False)[0]
print(f'\n  Matrix composition: B·A where A:{A.shape}, B:{B.shape}')
print(f'    L(A) = {L_A:.4f}, L(B) = {L_B:.4f}')
print(f'    L(BA) = {L_BA:.4f}, L(B)·L(A) = {L_B * L_A:.4f}')
print(f'    L(BA) ≤ L(B)·L(A): {L_BA <= L_B * L_A + 1e-10} ✓')

# ══════════════════════════════════════════════════════════════════
# (c) Bounded derivative implies Lipschitz (Mean Value Theorem)
# ══════════════════════════════════════════════════════════════════

print(f'\n(c) Bounded derivative ⟹ Lipschitz (Mean Value Theorem)')
print('-' * 60)

# Proof sketch:
# By MVT: f(x) - f(y) = f'(c)(x - y) for some c between x, y
# |f(x) - f(y)| = |f'(c)| · |x - y| ≤ (max|f'|) · |x - y|
# So L = max|f'(x)| = sup_x |f'(x)|

activations_lip = [
    ('ReLU',    lambda x: np.maximum(0, x),                 lambda x: np.where(x > 0, 1.0, 0.0),    1.0),
    ('Sigmoid', lambda x: 1/(1+np.exp(-np.clip(x,-500,500))), lambda x: sigmoid(x)*(1-sigmoid(x)),  0.25),
    ('Tanh',    np.tanh,                                     lambda x: 1 - np.tanh(x)**2,            1.0),
]

x_test = np.linspace(-10, 10, 500000)
for name, fn, dfn, theoretical_L in activations_lip:
    max_deriv = np.max(np.abs(dfn(x_test)))
    estimated_L = estimate_lipschitz_1d(fn, x_test)
    
    print(f'  {name}:')
    print(f'    max|f\'(x)| ≈ {max_deriv:.6f}  (theoretical: {theoretical_L})')
    print(f'    Estimated L ≈ {estimated_L:.6f}')
    print(f'    Match: {abs(max_deriv - theoretical_L) < 0.01} ✓')

# ══════════════════════════════════════════════════════════════════
# Bonus: Lipschitz constant of a 2-layer neural network
# ══════════════════════════════════════════════════════════════════

print(f'\n{"="*70}')
print('Bonus: Lipschitz constant of a 2-layer network')
print('  Network: f(x) = W₂ · ReLU(W₁ · x)')
print('  L(f) ≤ ||W₂||₂ · L(ReLU) · ||W₁||₂ = ||W₂||₂ · 1 · ||W₁||₂')

W1 = np.random.randn(10, 5) * 0.5
W2 = np.random.randn(3, 10) * 0.5

L_W1 = np.linalg.svd(W1, compute_uv=False)[0]
L_W2 = np.linalg.svd(W2, compute_uv=False)[0]
L_relu = 1.0
L_bound = L_W2 * L_relu * L_W1

# Estimate empirically
def two_layer_net(x):
    h = np.maximum(0, W1 @ x)  # ReLU(W1 x)
    return W2 @ h

n_pairs = 100000
X1 = np.random.randn(n_pairs, 5)
X2 = X1 + np.random.randn(n_pairs, 5) * 0.01  # small perturbations
Y1 = np.array([two_layer_net(x) for x in X1])
Y2 = np.array([two_layer_net(x) for x in X2])
ratios = np.linalg.norm(Y1 - Y2, axis=1) / (np.linalg.norm(X1 - X2, axis=1) + 1e-15)
L_empirical = ratios.max()

print(f'  ||W₁||₂ = {L_W1:.4f}')
print(f'  ||W₂||₂ = {L_W2:.4f}')
print(f'  Upper bound: {L_bound:.4f}')
print(f'  Empirical L:  {L_empirical:.4f}')
print(f'  Empirical ≤ Bound: {L_empirical <= L_bound + 0.01} ✓')
print(f'  → Spectral normalization (Miyato 2018) constrains these norms')

print('\nAll tests passed ✓')

---

## Exercise 6: Higher-Order Functions ⭐⭐

**Background**: A higher-order function takes a function as input and/or returns a function as output.
The three fundamental higher-order functions — **map**, **filter**, **reduce** — appear throughout 
neural network computation:

- **Map**: apply function elementwise → activation functions, layer normalization
- **Filter**: keep elements satisfying a predicate → attention masking, top-k sampling
- **Reduce**: combine elements with binary function → sum pooling, softmax denominator

**References**: Church (1936), lambda calculus; Bird & Wadler (1988), *Introduction to Functional Programming*.

**Task**: 
(a) Express "apply ReLU elementwise to vector x ∈ ℝⁿ" as a map operation
(b) Express "top-k sampling: keep only tokens with probability in top k" as filter
(c) Express "softmax denominator ∑ exp(zᵢ)" as reduce
(d) Express "attention output = weighted sum of values" as reduce
(e) Implement currying: convert f(x, y) = x·y into curry(f)(x)(y)

**Requirements**:
- Implement generic `my_map`, `my_filter`, `my_reduce` functions
- Use them to solve each sub-problem
- Show equivalence with numpy vectorized operations

In [ ]:
# Exercise 6: Your Solution

def my_map(f: Callable, lst: list) -> list:
    """Apply f to every element: [f(x₁), f(x₂), ..., f(xₙ)]."""
    # YOUR CODE HERE
    pass

def my_filter(predicate: Callable, lst: list) -> list:
    """Keep elements where predicate(x) is True."""
    # YOUR CODE HERE
    pass

def my_reduce(binary_fn: Callable, lst: list, initial):
    """Combine elements: binary_fn(...binary_fn(binary_fn(initial, x₁), x₂)..., xₙ)."""
    # YOUR CODE HERE
    pass

# (a) Apply ReLU elementwise using map
# YOUR CODE HERE

# (b) Top-k sampling using filter
# YOUR CODE HERE

# (c) Softmax denominator using reduce
# YOUR CODE HERE

# (d) Weighted sum (attention output) using reduce
# YOUR CODE HERE

# (e) Currying: convert f(x,y) into curry(f)(x)(y)
# YOUR CODE HERE

In [ ]:
# Exercise 6: Solution ✅

def my_map(f: Callable, lst: list) -> list:
    """Apply f to every element: [f(x₁), f(x₂), ..., f(xₙ)]."""
    return [f(x) for x in lst]

def my_filter(predicate: Callable, lst: list) -> list:
    """Keep elements where predicate(x) is True."""
    return [x for x in lst if predicate(x)]

def my_reduce(binary_fn: Callable, lst: list, initial):
    """Combine elements left to right: binary_fn(...binary_fn(initial, x₁), x₂)..., xₙ)."""
    acc = initial
    for x in lst:
        acc = binary_fn(acc, x)
    return acc

print('Exercise 6: Higher-Order Functions')
print('=' * 65)

# ══════════════════════════════════════════════════════════════════
# (a) Apply ReLU elementwise using map
# ══════════════════════════════════════════════════════════════════
print('\n(a) ReLU via map:')
x_vec = [-3.0, -1.0, 0.0, 0.5, 2.0, -0.7, 4.0]
relu_single = lambda x: max(0.0, x)
result_map = my_map(relu_single, x_vec)

# Verify against numpy
result_np = np.maximum(0, np.array(x_vec)).tolist()
assert result_map == result_np, 'Map ReLU != numpy ReLU'

print(f'  Input:           {x_vec}')
print(f'  map(ReLU, x):    {result_map}')
print(f'  np.maximum(0,x): {result_np}')
print(f'  Match ✓')
print(f'  → In PyTorch: F.relu(x) is map(relu, x) applied elementwise')

# ══════════════════════════════════════════════════════════════════
# (b) Top-k sampling using filter
# ══════════════════════════════════════════════════════════════════
print('\n(b) Top-k sampling via filter:')
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'ate']
probs = [0.35, 0.20, 0.15, 0.10, 0.08, 0.05, 0.04, 0.03]
token_probs = list(zip(vocab, probs))

k = 3
# Sort by probability, find top-k threshold
sorted_probs = sorted(probs, reverse=True)
threshold = sorted_probs[k - 1] if k <= len(sorted_probs) else 0.0

top_k = my_filter(lambda tp: tp[1] >= threshold, token_probs)

print(f'  All tokens:  {token_probs}')
print(f'  Top-{k} filter: {top_k}')
print(f'  → Only tokens with prob ≥ {threshold} survive')
print(f'  → In LLM inference: top-k sampling filters vocabulary before sampling')

# ══════════════════════════════════════════════════════════════════
# (c) Softmax denominator using reduce
# ══════════════════════════════════════════════════════════════════
print('\n(c) Softmax denominator via reduce:')
logits = [2.0, 1.0, 0.5, -1.0, 3.0]
import math
exp_logits = my_map(math.exp, logits)
softmax_denom = my_reduce(lambda acc, x: acc + x, exp_logits, 0.0)

# Verify against numpy
logits_np = np.array(logits)
denom_np = np.sum(np.exp(logits_np))

print(f'  Logits:            {logits}')
print(f'  map(exp, logits):  {[round(e, 4) for e in exp_logits]}')
print(f'  reduce(+, exps):   {softmax_denom:.6f}')
print(f'  np.sum(np.exp()):  {denom_np:.6f}')
assert abs(softmax_denom - denom_np) < 1e-10, 'Reduce != numpy'
print(f'  Match ✓')

# Full softmax = map(lambda z: exp(z)/denom, logits)
softmax_result = my_map(lambda z: math.exp(z) / softmax_denom, logits)
softmax_np = (np.exp(logits_np) / denom_np).tolist()
print(f'  Full softmax:      {[round(p, 4) for p in softmax_result]}')
print(f'  Sum = {sum(softmax_result):.6f} (should be 1.0)')
print(f'  → softmax = map(exp) then reduce(+) to get denom, then map(/denom)')

# ══════════════════════════════════════════════════════════════════
# (d) Attention weighted sum using reduce
# ══════════════════════════════════════════════════════════════════
print('\n(d) Attention output (weighted sum of values) via reduce:')
# attn_output = sum_i (weight_i * value_i)
weights = [0.5, 0.3, 0.2]
values = [np.array([1.0, 0.0, 1.0]),
          np.array([0.0, 1.0, 0.0]),
          np.array([1.0, 1.0, 0.0])]

weighted_pairs = list(zip(weights, values))
attn_output = my_reduce(
    lambda acc, wv: acc + wv[0] * wv[1],
    weighted_pairs,
    np.zeros(3)
)

# Verify against direct computation
direct = sum(w * v for w, v in zip(weights, values))
assert np.allclose(attn_output, direct), 'Reduce != direct'

print(f'  Weights: {weights}')
print(f'  Values:  {[v.tolist() for v in values]}')
print(f'  reduce(acc + w·v, pairs, 0): {attn_output}')
print(f'  Direct:                       {direct}')
print(f'  Match ✓')
print(f'  → Attention: softmax(QKᵀ/√d)V is reduce(weighted_sum) over value vectors')

# ══════════════════════════════════════════════════════════════════
# (e) Currying
# ══════════════════════════════════════════════════════════════════
print('\n(e) Currying:')

def curry(f: Callable) -> Callable:
    """Convert f(x, y) into curry(f)(x)(y)."""
    return lambda x: lambda y: f(x, y)

# Example: multiply
multiply = lambda x, y: x * y
curried_mul = curry(multiply)

# curry(multiply)(3) returns a function that multiplies by 3
triple = curried_mul(3)
print(f'  multiply(x, y) = x * y')
print(f'  curried = curry(multiply)')
print(f'  triple = curried(3)  →  triple is the function y ↦ 3y')
print(f'  triple(5) = {triple(5)}  (= 3 × 5 = 15)')
print(f'  triple(7) = {triple(7)}  (= 3 × 7 = 21)')
assert triple(5) == 15 and triple(7) == 21, 'Curry failed'

# AI application: attention score
# score(q, k) = qᵀk / √d
d_k = 64
score = lambda q, k: np.dot(q, k) / np.sqrt(d_k)
curried_score = curry(score)

query = np.random.randn(d_k)
score_for_query = curried_score(query)  # fix query, get function of keys

keys = [np.random.randn(d_k) for _ in range(5)]
scores = my_map(score_for_query, keys)

print(f'\n  AI application: curried attention score')
print(f'  score(q, k) = qᵀk / √d_k, with d_k = {d_k}')
print(f'  score_q = curry(score)(q)  →  score_q(k) = qᵀk / √{d_k}')
print(f'  Scores for 5 keys: {[round(s, 4) for s in scores]}')
print(f'  → Partial application: fix the query, map over all keys')

print('\nAll tests passed ✓')

---

## Exercise 7: Inverse Functions and Pseudo-Inverse ⭐⭐⭐

**Background**: A function f: A → B has:
- **Left inverse** g if g ∘ f = id_A (g "undoes" f; requires f injective)
- **Right inverse** h if f ∘ h = id_B (h "undoes" going backward; requires f surjective)
- **Two-sided inverse** f⁻¹ if f⁻¹ ∘ f = id_A and f ∘ f⁻¹ = id_B (requires f bijective)

For non-invertible matrices, the **Moore-Penrose pseudo-inverse** A⁺ provides the best 
least-squares solution. Via SVD: if A = UΣVᵀ then A⁺ = VΣ⁺Uᵀ.

**References**: 
- Moore (1920), Penrose (1955) — pseudo-inverse definition
- Strang, *Linear Algebra and Its Applications*, 5th ed. (2016) — SVD and pseudo-inverses
- Golub & Van Loan, *Matrix Computations*, 4th ed. (2013) — numerical computation of pseudo-inverse

**Task**:
(a) Show f(x) = ln(x) on (0,∞) and g(x) = eˣ on ℝ are mutual inverses
(b) Find the pseudo-inverse of A = [[1,0],[0,1],[0,0]] (3×2); verify all 4 Penrose conditions
(c) For f(x) = 2x + 3, construct f⁻¹; verify f⁻¹ ∘ f = id and f ∘ f⁻¹ = id
(d) Demonstrate (g ∘ f)⁻¹ = f⁻¹ ∘ g⁻¹ (reverse order!)

**Requirements**:
- Compute pseudo-inverse both via SVD and via formula (AᵀA)⁻¹Aᵀ
- Verify all 4 Penrose conditions for the pseudo-inverse
- Verify the reverse-order inverse of composition

In [ ]:
# Exercise 7: Your Solution

# (a) Show ln and exp are mutual inverses
# Verify: ln(exp(x)) = x for all x ∈ ℝ
# Verify: exp(ln(x)) = x for all x ∈ (0, ∞)
# YOUR CODE HERE

# (b) Pseudo-inverse of A = [[1,0],[0,1],[0,0]]
# Compute via SVD: A = UΣVᵀ → A⁺ = VΣ⁺Uᵀ
# Verify 4 Penrose conditions:
#   1. AA⁺A = A
#   2. A⁺AA⁺ = A⁺
#   3. (AA⁺)ᵀ = AA⁺
#   4. (A⁺A)ᵀ = A⁺A
# YOUR CODE HERE

# (c) f(x) = 2x + 3, find f⁻¹, verify round-trips
# YOUR CODE HERE

# (d) (g ∘ f)⁻¹ = f⁻¹ ∘ g⁻¹
# YOUR CODE HERE

In [ ]:
# Exercise 7: Solution ✅

print('Exercise 7: Inverse Functions and Pseudo-Inverse')
print('=' * 70)

# ══════════════════════════════════════════════════════════════════
# (a) ln and exp are mutual inverses
# ══════════════════════════════════════════════════════════════════
print('\n(a) f(x) = ln(x) and g(x) = eˣ are mutual inverses')
print('-' * 60)

# f: (0, ∞) → ℝ, f(x) = ln(x)
# g: ℝ → (0, ∞), g(x) = eˣ
# Need to verify: g(f(x)) = x for all x > 0  AND  f(g(x)) = x for all x ∈ ℝ

# Test g ∘ f = id on (0, ∞)
x_pos = np.array([0.001, 0.1, 0.5, 1.0, 2.71828, 10.0, 100.0, 1e6])
roundtrip_gf = np.exp(np.log(x_pos))  # exp(ln(x)) should equal x
print(f'  x:           {x_pos}')
print(f'  exp(ln(x)):  {roundtrip_gf}')
assert np.allclose(roundtrip_gf, x_pos), 'exp(ln(x)) ≠ x'
print(f'  exp(ln(x)) = x  ✓  (g ∘ f = id on (0,∞))')

# Test f ∘ g = id on ℝ
x_real = np.array([-100, -10, -1, 0, 1, 5, 20, 50])
roundtrip_fg = np.log(np.exp(x_real))  # ln(exp(x)) should equal x
print(f'\n  x:           {x_real}')
print(f'  ln(exp(x)):  {roundtrip_fg}')
assert np.allclose(roundtrip_fg, x_real), 'ln(exp(x)) ≠ x'
print(f'  ln(exp(x)) = x  ✓  (f ∘ g = id on ℝ)')

print(f'\n  ∴ ln and exp are two-sided inverses')
print(f'  ln: (0,∞) → ℝ is bijective with inverse exp: ℝ → (0,∞)')

# ══════════════════════════════════════════════════════════════════
# (b) Moore-Penrose pseudo-inverse of A = [[1,0],[0,1],[0,0]]
# ══════════════════════════════════════════════════════════════════
print(f'\n(b) Pseudo-inverse of A = [[1,0],[0,1],[0,0]]')
print('-' * 60)

A = np.array([[1, 0],
              [0, 1],
              [0, 0]], dtype=float)

print(f'  A (3×2):\n{A}')

# Method 1: Via SVD
U, S, Vt = np.linalg.svd(A, full_matrices=False)
# A⁺ = V Σ⁺ Uᵀ
S_pinv = np.diag(1.0 / S)  # invert nonzero singular values
A_pinv_svd = Vt.T @ S_pinv @ U.T

print(f'\n  SVD: U =\n{U}')
print(f'  Singular values: {S}')
print(f'  Vᵀ =\n{Vt}')
print(f'\n  A⁺ (via SVD) =\n{A_pinv_svd}')

# Method 2: Via formula (AᵀA)⁻¹Aᵀ (works when A has full column rank)
A_pinv_formula = np.linalg.inv(A.T @ A) @ A.T
print(f'  A⁺ (via (AᵀA)⁻¹Aᵀ) =\n{A_pinv_formula}')
assert np.allclose(A_pinv_svd, A_pinv_formula), 'SVD vs formula pseudo-inverse mismatch'
print(f'  Both methods agree ✓')

# Method 3: numpy built-in
A_pinv_np = np.linalg.pinv(A)
assert np.allclose(A_pinv_svd, A_pinv_np), 'SVD vs numpy pinv mismatch'
print(f'  numpy.linalg.pinv also agrees ✓')

# Verify all 4 Penrose conditions
A_plus = A_pinv_svd
print(f'\n  Penrose conditions:')

# 1. AA⁺A = A
cond1 = A @ A_plus @ A
assert np.allclose(cond1, A), 'Penrose condition 1 failed'
print(f'  1. AA⁺A = A:       {np.allclose(cond1, A)} ✓')

# 2. A⁺AA⁺ = A⁺
cond2 = A_plus @ A @ A_plus
assert np.allclose(cond2, A_plus), 'Penrose condition 2 failed'
print(f'  2. A⁺AA⁺ = A⁺:     {np.allclose(cond2, A_plus)} ✓')

# 3. (AA⁺)ᵀ = AA⁺ (symmetric)
cond3_lhs = A @ A_plus
assert np.allclose(cond3_lhs, cond3_lhs.T), 'Penrose condition 3 failed'
print(f'  3. (AA⁺)ᵀ = AA⁺:   {np.allclose(cond3_lhs, cond3_lhs.T)} ✓')

# 4. (A⁺A)ᵀ = A⁺A (symmetric)
cond4_lhs = A_plus @ A
assert np.allclose(cond4_lhs, cond4_lhs.T), 'Penrose condition 4 failed'
print(f'  4. (A⁺A)ᵀ = A⁺A:   {np.allclose(cond4_lhs, cond4_lhs.T)} ✓')

# Least squares: A⁺b gives minimum-norm least squares solution
b = np.array([1, 2, 3], dtype=float)
x_ls = A_plus @ b
print(f'\n  Least squares: A⁺b where b = {b}')
print(f'  x = A⁺b = {x_ls}')
print(f'  Ax = {A @ x_ls} (best approximation to b in column space of A)')
print(f'  Residual ||Ax - b|| = {np.linalg.norm(A @ x_ls - b):.4f}')
print(f'  → The [0,0] row of A means 3rd component of b cannot be matched')

# ══════════════════════════════════════════════════════════════════
# (c) f(x) = 2x + 3, construct inverse
# ══════════════════════════════════════════════════════════════════
print(f'\n(c) f(x) = 2x + 3, inverse f⁻¹(y) = (y − 3) / 2')
print('-' * 60)

# To find f⁻¹: solve y = 2x + 3 for x → x = (y-3)/2
f_c = lambda x: 2*x + 3
f_c_inv = lambda y: (y - 3) / 2

test_x = np.array([-10, -1, 0, 1, 5, 42, 100])
# Verify f⁻¹(f(x)) = x
assert np.allclose(f_c_inv(f_c(test_x)), test_x), 'f⁻¹∘f ≠ id'
print(f'  f⁻¹(f(x)) = x  ✓  for x = {test_x.tolist()}')

# Verify f(f⁻¹(y)) = y
test_y = f_c(test_x)
assert np.allclose(f_c(f_c_inv(test_y)), test_y), 'f∘f⁻¹ ≠ id'
print(f'  f(f⁻¹(y)) = y  ✓  for y = {test_y.tolist()}')
print(f'  Both compositions equal identity ✓')

# ══════════════════════════════════════════════════════════════════
# (d) (g ∘ f)⁻¹ = f⁻¹ ∘ g⁻¹  (reverse order!)
# ══════════════════════════════════════════════════════════════════
print(f'\n(d) Inverse of composition: (g ∘ f)⁻¹ = f⁻¹ ∘ g⁻¹')
print('-' * 60)

# f(x) = 2x + 3,  f⁻¹(y) = (y-3)/2
# g(x) = 5x - 1,  g⁻¹(y) = (y+1)/5
# (g ∘ f)(x) = g(2x+3) = 5(2x+3) - 1 = 10x + 14
# (g ∘ f)⁻¹(y) = (y - 14) / 10

f_d = lambda x: 2*x + 3
g_d = lambda x: 5*x - 1
f_d_inv = lambda y: (y - 3) / 2
g_d_inv = lambda y: (y + 1) / 5

gf = lambda x: g_d(f_d(x))                     # g ∘ f
gf_inv_direct = lambda y: (y - 14) / 10        # (g ∘ f)⁻¹ computed directly
gf_inv_reverse = lambda y: f_d_inv(g_d_inv(y))  # f⁻¹ ∘ g⁻¹ (reverse order)

test_vals = np.array([-100, -5, 0, 1, 10, 50, 200])

# Both should give the same result
direct_results = gf_inv_direct(test_vals)
reverse_results = gf_inv_reverse(test_vals)
assert np.allclose(direct_results, reverse_results), 'Direct vs reverse mismatch'

print(f'  f(x) = 2x+3,  g(x) = 5x-1')
print(f'  (g ∘ f)(x) = 10x + 14')
print(f'  (g ∘ f)⁻¹(y) = (y-14)/10  (computed directly)')
print(f'  f⁻¹ ∘ g⁻¹(y) = f⁻¹(g⁻¹(y)) = f⁻¹((y+1)/5) = ((y+1)/5 - 3)/2 = (y-14)/10')
print(f'  Match on test values: {np.allclose(direct_results, reverse_results)} ✓')

# Verify round-trip
assert np.allclose(gf_inv_direct(gf(test_vals)), test_vals), 'Round-trip failed'
print(f'  (g∘f)⁻¹((g∘f)(x)) = x  ✓')

print(f'\n  Analogy: putting on shoes then socks reversed by')
print(f'  taking off socks then shoes — reverse order!')
print(f'  AI: invertible network decoding = inverse of each layer in REVERSE order')

print('\nAll tests passed ✓')

---

## Exercise 8: Universal Approximation ⭐⭐⭐

**Background**: The Universal Approximation Theorem (Cybenko, 1989; Hornik, Stinchcombe & White, 1989) 
states that a single hidden layer neural network with sufficient neurons and a non-polynomial 
activation function can approximate any continuous function on a compact set to arbitrary precision.

**Formal statement**: For any continuous f: [0,1]ⁿ → ℝ and any ε > 0, there exists N ∈ ℕ and 
parameters w, b, α such that:

$$\sup_{x \in [0,1]^n} \left| f(x) - \sum_{i=1}^{N} \alpha_i \sigma(w_i^T x + b_i) \right| < \varepsilon$$

where σ is any non-polynomial activation (sigmoid, ReLU, etc.).

**What the theorem guarantees**: existence of such a network
**What it does NOT guarantee**: 
- How large N needs to be (could be astronomically large)
- How to find the parameters w, b, α (no constructive proof)
- That gradient descent will find them (optimisation ≠ existence)
- Anything about generalization from finite samples

**Reference**: Hornik, Stinchcombe & White, *Multilayer feedforward networks are universal approximators*, 
Neural Networks (1989); Cybenko, *Approximation by superpositions of a sigmoidal function* (1989);
Lu et al., *The expressive power of neural networks* (2017) — depth separation results.

**Task**:
(a) State what UAT guarantees and what it does NOT (answer in comments)
(b) Build a 1-hidden-layer network with sigmoid activation that approximates f(x) = x² on [-1, 1] 
    to within ε = 0.1 by training with gradient descent
(c) Experimentally show the depth-width tradeoff: compare shallow-wide vs deep-narrow networks 
    approximating a target function

In [ ]:
# Exercise 8: Your Solution
# ========================

# (a) State what UAT guarantees vs. does NOT guarantee
# GUARANTEES:
# - ???
# DOES NOT GUARANTEE:
# - ???

# (b) Approximate f(x) = x² on [-1,1] with 1-hidden-layer sigmoid network
def sigmoid(z):
    pass  # Implement sigmoid activation

def shallow_network(x, W, b, alpha, beta):
    """1-hidden-layer network: alpha @ sigmoid(W @ x + b) + beta"""
    pass  # Implement forward pass

# Train parameters to get max |f̂(x) - x²| < 0.1 on [-1, 1]
# W: shape (N, 1), b: shape (N,), alpha: shape (N,), beta: scalar
# Hint: Start with N=10 neurons, lr=0.01, 5000 epochs

# (c) Depth-width tradeoff
# Compare: shallow network (1 layer, 64 neurons) vs deep network (4 layers, 8 neurons each)
# Target: f(x) = |sin(5x)| on [-1, 1] (highly oscillatory — hard for shallow nets)
# Track training loss over epochs for both

In [ ]:
# Exercise 8: Solution ✅
# ======================

# ===== (a) WHAT UAT GUARANTEES vs DOES NOT =====
print("=" * 65)
print("UNIVERSAL APPROXIMATION THEOREM — GUARANTEES vs LIMITATIONS")
print("=" * 65)

guarantees = [
    "Existence: for any continuous f on compact set and any ε>0,",
    "  there exists a 1-hidden-layer network within ε of f",
    "Generality: works for ANY continuous function",
    "Activation flexibility: works for sigmoid, ReLU, tanh, etc.",
    "  (any non-polynomial continuous activation suffices)",
]

does_not = [
    "Network SIZE: N (# neurons) may need to be exponentially large",
    "LEARNABILITY: no guarantee gradient descent finds good params",
    "GENERALIZATION: approximation on training data ≠ test data",
    "CONSTRUCTIVE: theorem is existential — no algorithm to build",
    "EFFICIENCY: a deep network may need exponentially fewer params",
    "SAMPLE COMPLEXITY: says nothing about how much data is needed",
]

print("\n✅ GUARANTEES:")
for g in guarantees:
    print(f"   • {g}")
print("\n❌ DOES NOT GUARANTEE:")
for d in does_not:
    print(f"   • {d}")

# ===== (b) Approximate f(x) = x² with shallow sigmoid network =====
print("\n" + "=" * 65)
print("PART (b): Shallow Network Approximating f(x) = x²")
print("=" * 65)

np.random.seed(42)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

# Network: f̂(x) = Σᵢ αᵢ σ(wᵢx + bᵢ) + β
N_neurons = 20  # Number of hidden neurons
x_train = np.linspace(-1, 1, 200).reshape(-1, 1)  # (200, 1)
y_train = x_train ** 2  # Target: x²

# Initialize parameters (Xavier-like)
W = np.random.randn(N_neurons, 1) * 1.0     # (N, 1)
b = np.random.randn(N_neurons) * 0.5         # (N,)
alpha = np.random.randn(N_neurons) * 0.1     # (N,)
beta = 0.0                                    # scalar bias

lr = 0.01
epochs = 8000
losses = []

for epoch in range(epochs):
    # Forward pass: x_train is (200,1), W is (N,1)
    z = x_train @ W.T + b  # (200, N)  — pre-activation
    h = sigmoid(z)          # (200, N)  — hidden activations
    y_pred = h @ alpha + beta  # (200,)  — output

    # Loss: MSE
    error = y_pred.reshape(-1, 1) - y_train  # (200, 1)
    loss = np.mean(error ** 2)
    losses.append(loss)

    # Backward pass
    dL_dy = 2 * error / len(x_train)  # (200, 1)

    # Gradients
    d_alpha = (h.T @ dL_dy).flatten()           # (N,)
    d_beta = np.sum(dL_dy)                       # scalar

    dL_dh = dL_dy @ alpha.reshape(1, -1)         # (200, N)
    dL_dz = dL_dh * sigmoid_deriv(z)             # (200, N)

    d_W = dL_dz.T @ x_train                     # (N, 1)
    d_b = np.sum(dL_dz, axis=0)                  # (N,)

    # Update
    alpha -= lr * d_alpha
    beta  -= lr * d_beta
    W     -= lr * d_W
    b     -= lr * d_b

    if epoch % 2000 == 0:
        print(f"  Epoch {epoch:5d}: MSE = {loss:.6f}")

# Final evaluation
z_final = x_train @ W.T + b
h_final = sigmoid(z_final)
y_final = h_final @ alpha + beta

max_error = np.max(np.abs(y_final.flatten() - y_train.flatten()))
print(f"\n  Final MSE: {losses[-1]:.6f}")
print(f"  Max |f̂(x) - x²|: {max_error:.4f}")
print(f"  Target ε = 0.1 → {'✅ ACHIEVED' if max_error < 0.1 else '❌ Not yet'}")
print(f"  Neurons used: {N_neurons}")

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Approximation
axes[0].plot(x_train, y_train, 'b-', linewidth=2, label='f(x) = x²')
axes[0].plot(x_train, y_final, 'r--', linewidth=2, label=f'Network ({N_neurons} neurons)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('Universal Approximation of x²')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Plot 2: Error
axes[1].plot(x_train, np.abs(y_final.flatten() - y_train.flatten()), 'g-', linewidth=1.5)
axes[1].axhline(y=0.1, color='r', linestyle='--', label='ε = 0.1')
axes[1].set_xlabel('x'); axes[1].set_ylabel('|error|')
axes[1].set_title('Approximation Error')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Plot 3: Training loss
axes[2].semilogy(losses)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MSE (log)')
axes[2].set_title('Training Loss')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exercise8_universal_approximation.png', dpi=100, bbox_inches='tight')
plt.show()

# ===== (c) Depth-Width Tradeoff =====
print("\n" + "=" * 65)
print("PART (c): Depth-Width Tradeoff")
print("=" * 65)
print("Comparing shallow-wide vs deep-narrow on f(x) = |sin(5x)|")

# Target: f(x) = |sin(5x)| — oscillatory, benefits from depth
x_data = np.linspace(-1, 1, 300).reshape(-1, 1)
y_data = np.abs(np.sin(5 * x_data))

def train_shallow(x, y, n_hidden, epochs=5000, lr=0.01):
    """1-hidden-layer network with n_hidden neurons."""
    np.random.seed(0)
    W1 = np.random.randn(n_hidden, 1) * 1.0
    b1 = np.random.randn(n_hidden) * 0.5
    W2 = np.random.randn(n_hidden) * 0.1
    b2 = 0.0
    losses = []
    for ep in range(epochs):
        z1 = x @ W1.T + b1
        h1 = np.maximum(z1, 0)  # ReLU
        out = h1 @ W2 + b2
        err = out.reshape(-1, 1) - y
        loss = np.mean(err ** 2)
        losses.append(loss)
        dout = 2 * err / len(x)
        dW2 = (h1.T @ dout).flatten()
        db2 = np.sum(dout)
        dh1 = dout @ W2.reshape(1, -1)
        dz1 = dh1 * (z1 > 0).astype(float)
        dW1 = dz1.T @ x
        db1 = np.sum(dz1, axis=0)
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
    z1 = x @ W1.T + b1; h1 = np.maximum(z1, 0)
    pred = h1 @ W2 + b2
    return losses, pred, n_hidden  # total params = 3*n_hidden + 1

def train_deep(x, y, layer_sizes, epochs=5000, lr=0.005):
    """Multi-layer ReLU network."""
    np.random.seed(0)
    layers = len(layer_sizes)
    Ws = []
    bs = []
    in_dim = 1
    for ls in layer_sizes:
        Ws.append(np.random.randn(ls, in_dim) * np.sqrt(2.0 / in_dim))
        bs.append(np.zeros(ls))
        in_dim = ls
    # Output layer
    W_out = np.random.randn(in_dim) * np.sqrt(2.0 / in_dim)
    b_out = 0.0
    losses = []
    for ep in range(epochs):
        # Forward
        activations = [x]
        pre_acts = []
        h = x
        for i in range(layers):
            z = h @ Ws[i].T + bs[i]
            pre_acts.append(z)
            h = np.maximum(z, 0)
            activations.append(h)
        out = h @ W_out + b_out
        err = out.reshape(-1, 1) - y
        loss = np.mean(err ** 2)
        losses.append(loss)
        # Backward
        dout = 2 * err / len(x)
        dW_out = (activations[-1].T @ dout).flatten()
        db_out = np.sum(dout)
        dh = dout @ W_out.reshape(1, -1)
        for i in range(layers - 1, -1, -1):
            dz = dh * (pre_acts[i] > 0).astype(float)
            dW = dz.T @ activations[i]
            db = np.sum(dz, axis=0)
            if i > 0:
                dh = dz @ Ws[i]
            Ws[i] -= lr * dW
            bs[i] -= lr * db
        W_out -= lr * dW_out
        b_out -= lr * db_out
    h = x
    for i in range(layers):
        h = np.maximum(h @ Ws[i].T + bs[i], 0)
    pred = h @ W_out + b_out
    total_params = sum(w.size + b.size for w, b in zip(Ws, bs)) + W_out.size + 1
    return losses, pred, total_params

# Train both architectures
# Shallow: 1 layer × 64 neurons = 64×1 + 64 + 64 + 1 = 193 params
losses_shallow, pred_shallow, params_s = train_shallow(x_data, y_data, 64, epochs=6000, lr=0.01)
print(f"  Shallow (1×64): {3*64+1} params, final MSE = {losses_shallow[-1]:.6f}")

# Deep: 4 layers × 8 neurons = much fewer params but more expressive per param
losses_deep, pred_deep, params_d = train_deep(x_data, y_data, [8, 8, 8, 8], epochs=6000, lr=0.005)
print(f"  Deep (4×8): {params_d} params, final MSE = {losses_deep[-1]:.6f}")

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(x_data, y_data, 'b-', linewidth=2, label='|sin(5x)|', alpha=0.7)
axes[0].plot(x_data, pred_shallow, 'r--', linewidth=1.5, label='Shallow 1×64')
axes[0].plot(x_data, pred_deep, 'g--', linewidth=1.5, label='Deep 4×8')
axes[0].set_title('Depth-Width Tradeoff')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(losses_shallow, 'r-', alpha=0.7, label='Shallow')
axes[1].semilogy(losses_deep, 'g-', alpha=0.7, label='Deep')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE (log)')
axes[1].set_title('Training Loss Comparison')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(x_data, np.abs(pred_shallow.flatten() - y_data.flatten()), 'r-', alpha=0.7, label='Shallow')
axes[2].plot(x_data, np.abs(pred_deep.flatten() - y_data.flatten()), 'g-', alpha=0.7, label='Deep')
axes[2].set_xlabel('x'); axes[2].set_ylabel('|error|')
axes[2].set_title('Pointwise Error Comparison')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exercise8_depth_width_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n📌 Key insight (Telgarsky 2016, Eldan & Shamir 2016):")
print("   There exist functions that a depth-k network represents with")
print("   O(poly(n)) neurons, but a depth-(k-1) network needs exp(n).")
print("   → Depth provides EXPONENTIAL efficiency gains for certain functions.")
print("   This is WHY modern AI uses deep networks, not wide shallow ones.")